In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np

from utils.plots import calculate_metrics, calculate_accuracy
from utils.tools import get_config, collect_jobs, prettify_table, format_table, aggregate_results
from utils.constants import PRED_COLS

In [3]:
cols = [
    "model",
    "use_instructions",
    "type_of_demonstrations",
    "number_of_demonstrations",
    "num_rows",
    "num_parse_errors",
    "num_gen_errors",
    #*PRED_COLS
]

def error_stats_table(jobs, cols):

    print(f"Processing {len(jobs)} jobs.")
    
    table = {col_name: [] for col_name in cols}
    for job in jobs:
        df = pd.read_csv(job["result_csv"], sep=";")
        num_rows = len(df)

        # LLM failed to produce proper json
        mask_no_errors = (df[PRED_COLS] != "\"PARSE ERROR\"").all(axis=1)
        df = df[mask_no_errors]
        num_parse_errors = list(mask_no_errors).count(False)

        # LLM did not follow instructions
        mask_generate_fail = df[PRED_COLS].isin(["\"yes\"", "\"no\""]).all(axis=1)
        df = df[mask_generate_fail]
        num_gen_errors = list(mask_generate_fail).count(False)

        config = get_config(job["config_json"])

        # Append run configs
        table["model"].append(job["model"])
        table["use_instructions"].append(config["use_instructions"])
        table["type_of_demonstrations"].append(config["type_of_demonstrations"])
        table["number_of_demonstrations"].append(config["number_of_demonstrations"])

        # Append error counts
        table["num_rows"].append(num_rows)
        table["num_parse_errors"].append(num_parse_errors)
        table["num_gen_errors"].append(num_gen_errors)

    return table

In [4]:
# Collect raw data
basepath = "./outputs/v6"
finished_jobs = collect_jobs(basepath)
jobs_list = [job for _, job_list in finished_jobs.items() for job in job_list] # Flatten

res = pd.DataFrame(error_stats_table(jobs_list, cols))
res = prettify_table(res)

Processing 330 jobs.


In [5]:
# Show all columns
pd.options.display.max_columns = None
# Set float formatting
pd.options.display.float_format = '{:,.3f}'.format
bycols = [
    "model",
    "number_of_demonstrations",
    "type_of_demonstrations",
    "use_instructions"
]

agg = aggregate_results(res, bycols, cols[5:], funs=["sum"])
# Drop count
agg = agg.loc[pd.IndexSlice[:, :, :, :], (slice(None))]
# Drop rows with no errors
error_rows = (agg.loc[:, :] > 0).any(axis=1)
agg = agg[error_rows]

In [6]:
formatted = format_table(agg)

formatted.columns = formatted.columns.get_level_values(0) # Flatten

parse_errs_sum = formatted["num parse errors"].sum()
gen_errs_sum = formatted["num gen errors"].sum()

formatted.loc[("Total sum", "", "", "")] = (parse_errs_sum, gen_errs_sum) # Add total

formatted

num parse errors  \
model       number of demonstrations type of demonstrations use instructions                     
Llama-8B    1                        negative               no                               5   
                                     positive               no                             508   
            6                        mixed                  no                              54   
                                     negative               no                               3   
                                     positive               no                             219   
Llama-70B   1                        negative               no                               5   
                                     positive               no                              11   
            6                        positive               no                               2   
Mistral-7B  1                        negative               no                             121   
                                     positive               no                              32   
            6                        mixed                  no                               1   
                                     negative               no                               3   
                                     positive               no                               1   
Mistral-24B 0                        none                   yes                              3   
            1                        negative               no                              22   
                                     positive               no                               6   
            6                        mixed                  no                              15   
                                                            yes                              5   
                                     negative               no                              22   
                                                            yes                              7   
                                     positive               no                               7   
                                                            yes                              1   
Qwen3-32B   1                        negative               no                               0   
            6                        negative               no                               0   
                                                            yes                              1   
Total sum                                                                                 1054   

                                                                              num gen errors  
model       number of demonstrations type of demonstrations use instructions                  
Llama-8B    1                        negative               no                             0  
                                     positive               no                             0  
            6                        mixed                  no                             0  
                                     negative               no                             0  
                                     positive               no                             0  
Llama-70B   1                        negative               no                             0  
                                     positive               no                             0  
            6                        positive               no                             0  
Mistral-7B  1                        negative               no                             0  
                                     positive               no                             0  
            6                        mixed                  no                             0  
                                     negative               no                             0  
   

In [8]:
tables_location = "./latex/tables"
os.makedirs(tables_location, exist_ok=True)
with open(f"{tables_location}/err_table.tex", "w", encoding="UTF-8") as f:
    text = formatted.to_latex(escape=True, sparsify=True)
    #f.write(text)